# Machinery Rentals — Churn Prediction Model

**Two-tier analysis:**
- **Model A** — Full customer base (1,621 customers)
- **Model B** — Top 177 customers (Pareto 80/20: ~11% of customers driving ~80% of revenue)

**Churn definition:** A customer is flagged as churned if their last rental transaction is more than 90 days before the reference date (end of dataset).

**Features used:**
- RFM: Recency, Frequency, Monetary
- Regularity: CV of Interpurchase Time and CV of Contract Duration (Erlang-k derived)

**Output:** Churn probability score + risk segment (High / Medium / Low) per customer

---
## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, classification_report, roc_curve,
    ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'high': '#d62728', 'medium': '#ff7f0e', 'low': '#2ca02c', 'neutral': '#1f77b4'}
print('Libraries loaded.')

---
## 1. Load & Clean Data

In [ ]:
df = pd.read_csv('Machinery_dataset_service.csv')
print(f'Raw shape: {df.shape}')
print(f'Unique customers: {df["Customer ID"].nunique()}')

# Parse dates — Factor Date is the billing/invoice date (most reliable)
df['Factor Date'] = pd.to_datetime(df['Factor Date'], format='%Y/%m/%d', errors='coerce')
df['Start of rental'] = pd.to_datetime(df['Start of rental'], errors='coerce')
df['End of rental'] = pd.to_datetime(df['End of rental'], errors='coerce')

# Remove rows with missing dates or negative amounts (credit notes / corrections)
df = df.dropna(subset=['Factor Date', 'Customer ID'])
df = df[df['Amount'] > 0]

# Filter to valid date window (2021-2024 per Factor Date)
df = df[(df['Factor Date'].dt.year >= 2021) & (df['Factor Date'].dt.year <= 2024)]

print(f'Clean shape: {df.shape}')
print(f'Date range: {df["Factor Date"].min().date()} → {df["Factor Date"].max().date()}')

---
## 2. Build RFM Features

In [ ]:
REFERENCE_DATE = df['Factor Date'].max()
print(f'Reference date (snapshot): {REFERENCE_DATE.date()}')

rfm_customer = df.groupby('Customer ID').agg(
    Last_Purchase=('Factor Date', 'max'),
    Frequency=('Factor Date', 'count'),
    Monetary=('Amount', 'sum')
).reset_index()

rfm_customer['Recency'] = (REFERENCE_DATE - rfm_customer['Last_Purchase']).dt.days
print(rfm_customer[['Recency', 'Frequency', 'Monetary']].describe().round(1))

---
## 3. Build Regularity Features (CV Interpurchase Time & CV Contract Duration)

In [ ]:
# --- CV of Interpurchase Time ---
# Sort transactions per customer by date, compute gaps between consecutive purchases
df_sorted = df.sort_values(['Customer ID', 'Factor Date'])
df_sorted['Prev_Date'] = df_sorted.groupby('Customer ID')['Factor Date'].shift(1)
df_sorted['Interpurchase_Days'] = (df_sorted['Factor Date'] - df_sorted['Prev_Date']).dt.days

customer_time_regularity = df_sorted.groupby('Customer ID')['Interpurchase_Days'].agg(
    mean_ipt='mean',
    std_ipt='std'
).reset_index()
customer_time_regularity['CV_Interpurchase_Time (Cust.)'] = (
    customer_time_regularity['std_ipt'] / customer_time_regularity['mean_ipt']
).fillna(0)

# --- CV of Contract Duration ---
# Contract duration = End of rental - Start of rental in days
df['Contract_Duration_Days'] = (df['End of rental'] - df['Start of rental']).dt.days
df_contracts = df[df['Contract_Duration_Days'] > 0].copy()

quantity_regularity_customer = df_contracts.groupby('Customer ID')['Contract_Duration_Days'].agg(
    mean_dur='mean',
    std_dur='std'
).reset_index()
quantity_regularity_customer['CV_Contract_Duration (Cust.)'] = (
    quantity_regularity_customer['std_dur'] / quantity_regularity_customer['mean_dur']
).fillna(0)

print('CV Interpurchase Time — sample:')
print(customer_time_regularity[['Customer ID','CV_Interpurchase_Time (Cust.)']].head())
print('\nCV Contract Duration — sample:')
print(quantity_regularity_customer[['Customer ID','CV_Contract_Duration (Cust.)']].head())

---
## 4. Assemble Master Feature Table

In [ ]:
eps = 1e-6

rfm = rfm_customer.copy()

# Churn label: inactive for > 90 days
rfm['churn_90'] = (rfm['Recency'] > 90).astype(int)

# Merge regularity CVs
rfm = rfm.merge(
    customer_time_regularity[['Customer ID','CV_Interpurchase_Time (Cust.)']],
    on='Customer ID', how='left'
).merge(
    quantity_regularity_customer[['Customer ID','CV_Contract_Duration (Cust.)']],
    on='Customer ID', how='left'
).dropna(subset=['CV_Interpurchase_Time (Cust.)','CV_Contract_Duration (Cust.)'])

# Erlang-k shape parameters
rfm['CV_time'] = rfm['CV_Interpurchase_Time (Cust.)'].replace(0, eps)
rfm['CV_dur']  = rfm['CV_Contract_Duration (Cust.)'].replace(0, eps)
rfm['k_time']  = 1 / (rfm['CV_time'] ** 2)
rfm['k_dur']   = 1 / (rfm['CV_dur']  ** 2)

print(f'Master table: {rfm.shape[0]} customers')
print(f'Churn balance:\n{rfm["churn_90"].value_counts().rename({0:"Active",1:"Churned"})}')

---
## 5. Helper Functions

In [ ]:
FEATURES = [
    'Frequency', 'Monetary',
    'CV_Interpurchase_Time (Cust.)', 'CV_Contract_Duration (Cust.)',
    'k_time', 'k_dur'
]

def train_and_evaluate(data, label, features=FEATURES):
    """Train GBC, evaluate, return model + scored dataframe."""
    X = data[features]
    y = data['churn_90']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    clf = GradientBoostingClassifier(random_state=42)
    clf.fit(X_train, y_train)

    y_prob = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)

    print(f'\n=== {label} ===')
    print(f'Customers in model : {len(data)}')
    print(f'ROC AUC            : {auc:.4f}')
    print(classification_report(y_test, clf.predict(X_test), target_names=['Active','Churned']))

    # Score all customers
    data = data.copy()
    data['churn_prob'] = clf.predict_proba(X)[:, 1]
    data['risk_segment'] = pd.cut(
        data['churn_prob'],
        bins=[0, 0.35, 0.65, 1.0],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    return clf, data, (X_test, y_test, y_prob), auc


def plot_results(clf, eval_tuple, data, label, features=FEATURES):
    """4-panel diagnostic plot."""
    X_test, y_test, y_prob = eval_tuple
    auc = roc_auc_score(y_test, y_prob)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{label} — Churn Model Diagnostics', fontsize=15, fontweight='bold')

    # 1) ROC Curve
    ax = axes[0, 0]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, color=COLORS['neutral'], lw=2, label=f'AUC = {auc:.2f}')
    ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve'); ax.legend()

    # 2) Precision-Recall Curve
    ax = axes[0, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(rec, prec, color=COLORS['high'], lw=2, label=f'AP = {ap:.2f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve'); ax.legend()

    # 3) Feature Importances
    ax = axes[1, 0]
    imp = pd.Series(clf.feature_importances_, index=features).sort_values()
    imp.plot(kind='barh', ax=ax, color=COLORS['neutral'])
    ax.set_title('Feature Importances'); ax.set_xlabel('Importance')

    # 4) Risk Segment Distribution
    ax = axes[1, 1]
    seg_counts = data['risk_segment'].value_counts().reindex(['High','Medium','Low'])
    bars = ax.bar(seg_counts.index, seg_counts.values,
                  color=[COLORS['high'], COLORS['medium'], COLORS['low']])
    for bar, val in zip(bars, seg_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontweight='bold')
    ax.set_title('Risk Segment Distribution')
    ax.set_ylabel('Number of Customers')

    plt.tight_layout()
    plt.savefig(f'{label.replace(" ","_")}_diagnostics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved.')

print('Helper functions defined.')

---
## 6. MODEL A — Full Customer Base

In [ ]:
clf_all, rfm_all_scored, eval_all, auc_all = train_and_evaluate(
    rfm, label='Model A — Full Customer Base'
)

In [ ]:
plot_results(clf_all, eval_all, rfm_all_scored, label='Model A — Full Customer Base')

In [ ]:
# Output table — all customers scored
output_all = rfm_all_scored[[
    'Customer ID', 'Recency', 'Frequency', 'Monetary',
    'CV_Interpurchase_Time (Cust.)', 'CV_Contract_Duration (Cust.)',
    'churn_90', 'churn_prob', 'risk_segment'
]].sort_values('churn_prob', ascending=False).reset_index(drop=True)

output_all['churn_prob'] = output_all['churn_prob'].round(4)
print('Top 20 highest churn risk customers:')
output_all.head(20)

In [ ]:
# Segment summary
seg_summary_all = rfm_all_scored.groupby('risk_segment', observed=True).agg(
    Customers=('Customer ID','count'),
    Avg_Churn_Prob=('churn_prob','mean'),
    Avg_Monetary=('Monetary','mean'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean')
).round(1).reindex(['High','Medium','Low'])

print('\n=== Model A — Segment Summary ===')
seg_summary_all

---
## 7. Identify Top 177 Customers (Pareto 80/20)

In [ ]:
# Sort customers by total revenue descending
revenue_rank = rfm[['Customer ID','Monetary']].sort_values('Monetary', ascending=False).reset_index(drop=True)
revenue_rank['cumulative_revenue'] = revenue_rank['Monetary'].cumsum()
revenue_rank['cumulative_pct'] = revenue_rank['cumulative_revenue'] / revenue_rank['Monetary'].sum() * 100

# Find exact cutoff
top177_ids = revenue_rank.head(177)['Customer ID'].tolist()
actual_pct = revenue_rank.head(177)['cumulative_pct'].iloc[-1]

print(f'Top 177 customers represent {actual_pct:.1f}% of total revenue')
print(f'Total revenue (all): €{revenue_rank["Monetary"].sum():,.0f}')
print(f'Revenue (top 177):   €{revenue_rank.head(177)["Monetary"].sum():,.0f}')

# Pareto chart
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(range(len(revenue_rank)), revenue_rank['Monetary'], color=COLORS['neutral'], alpha=0.6, label='Revenue')
ax1.axvline(x=177, color=COLORS['high'], linestyle='--', lw=2, label='Top 177 cutoff')
ax1.set_xlabel('Customer Rank'); ax1.set_ylabel('Total Revenue (€)')

ax2 = ax1.twinx()
ax2.plot(range(len(revenue_rank)), revenue_rank['cumulative_pct'],
         color=COLORS['medium'], lw=2, label='Cumulative %')
ax2.axhline(y=80, color='grey', linestyle=':', lw=1)
ax2.set_ylabel('Cumulative Revenue (%)')
ax2.set_ylim(0, 105)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.title('Pareto Chart — Customer Revenue Distribution')
plt.tight_layout()
plt.savefig('pareto_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. MODEL B — Top 177 Customers

In [ ]:
rfm_top177 = rfm[rfm['Customer ID'].isin(top177_ids)].copy()

print(f'Top 177 subset shape: {rfm_top177.shape}')
print(f'Churn balance (Top 177):\n{rfm_top177["churn_90"].value_counts().rename({0:"Active",1:"Churned"})}')

In [ ]:
clf_top, rfm_top_scored, eval_top, auc_top = train_and_evaluate(
    rfm_top177, label='Model B — Top 177 Customers'
)

In [ ]:
plot_results(clf_top, eval_top, rfm_top_scored, label='Model B — Top 177 Customers')

In [ ]:
# Output table — top 177 scored
output_top177 = rfm_top_scored[[
    'Customer ID', 'Recency', 'Frequency', 'Monetary',
    'CV_Interpurchase_Time (Cust.)', 'CV_Contract_Duration (Cust.)',
    'churn_90', 'churn_prob', 'risk_segment'
]].sort_values('churn_prob', ascending=False).reset_index(drop=True)

output_top177['churn_prob'] = output_top177['churn_prob'].round(4)
print('Top 20 highest churn risk — among top 177:')
output_top177.head(20)

In [ ]:
# Segment summary — Top 177
seg_summary_top = rfm_top_scored.groupby('risk_segment', observed=True).agg(
    Customers=('Customer ID','count'),
    Avg_Churn_Prob=('churn_prob','mean'),
    Avg_Monetary=('Monetary','mean'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean')
).round(1).reindex(['High','Medium','Low'])

print('\n=== Model B — Segment Summary (Top 177) ===')
seg_summary_top

---
## 9. Side-by-Side Comparison: All Customers vs Top 177

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model A (All) vs Model B (Top 177) — Comparison', fontsize=14, fontweight='bold')

# AUC comparison
ax = axes[0]
ax.bar(['All Customers', 'Top 177'], [auc_all, auc_top],
       color=[COLORS['neutral'], COLORS['high']], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel('ROC AUC')
ax.set_title('Model Performance (AUC)')
for i, v in enumerate([auc_all, auc_top]):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Risk distribution — All
ax = axes[1]
seg_a = rfm_all_scored['risk_segment'].value_counts().reindex(['High','Medium','Low'])
ax.bar(seg_a.index, seg_a.values, color=[COLORS['high'], COLORS['medium'], COLORS['low']])
ax.set_title('Risk Segments — All Customers')
ax.set_ylabel('Count')
for i, v in enumerate(seg_a.values):
    ax.text(i, v + 1, str(v), ha='center', fontweight='bold')

# Risk distribution — Top 177
ax = axes[2]
seg_b = rfm_top_scored['risk_segment'].value_counts().reindex(['High','Medium','Low'])
ax.bar(seg_b.index, seg_b.values, color=[COLORS['high'], COLORS['medium'], COLORS['low']])
ax.set_title('Risk Segments — Top 177')
ax.set_ylabel('Count')
for i, v in enumerate(seg_b.values):
    ax.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Export Results

In [ ]:
# Export both scored tables to CSV
output_all.to_csv('churn_scores_all_customers.csv', index=False)
output_top177.to_csv('churn_scores_top177.csv', index=False)

print('Exported:')
print('  churn_scores_all_customers.csv')
print('  churn_scores_top177.csv')
print()
print('=== FINAL SUMMARY ===')
print(f'Model A — All customers  | AUC: {auc_all:.3f} | N: {len(rfm_all_scored)}')
print(f'Model B — Top 177        | AUC: {auc_top:.3f} | N: {len(rfm_top_scored)}')
print()
print('--- All Customers Risk Breakdown ---')
print(rfm_all_scored['risk_segment'].value_counts().reindex(['High','Medium','Low']).to_string())
print()
print('--- Top 177 Risk Breakdown ---')
print(rfm_top_scored['risk_segment'].value_counts().reindex(['High','Medium','Low']).to_string())